# BTCUSDT Microstructure — One Click Collector

**Fixed version:** Binance futures `metrics` archives are downloaded from the daily path. The previous notebook incorrectly requested monthly metrics files, which caused the 404 errors.

Run **Runtime → Run all**. No local Python installation is required.


In [ ]:
!pip -q install -U pandas pyarrow requests tqdm python-dateutil


In [ ]:
import os, time, json, zipfile, hashlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
import pandas as pd

START = pd.Timestamp(os.environ.get('START_DATE','2025-01-01'), tz='UTC')
END   = pd.Timestamp(os.environ.get('END_DATE','2026-05-31'), tz='UTC')
SYMBOL='BTCUSDT'
BASE='https://data.binance.vision/data/futures/um'
ROOT=Path('/content/btcusdt_research_v2'); CACHE=ROOT/'cache'; RAW=ROOT/'raw'; CAN=ROOT/'canonical'; REP=ROOT/'reports'
for p in (CACHE,RAW,CAN,REP): p.mkdir(parents=True,exist_ok=True)
S=requests.Session(); S.headers.update({'User-Agent':'BTCUSDT-research-collector/2.0'})

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''): h.update(b)
    return h.hexdigest()

def iter_months(a,b):
    x=a.normalize().replace(day=1); out=[]
    while x<=b: out.append(x); x=x+pd.offsets.MonthBegin(1)
    return out

def iter_days(a,b):
    x=a.normalize(); out=[]
    while x<=b: out.append(x); x=x+pd.Timedelta(days=1)
    return out

def download(url,target,retries=7):
    target=Path(target); target.parent.mkdir(parents=True,exist_ok=True); part=Path(str(target)+'.part')
    for attempt in range(retries):
        try:
            start=part.stat().st_size if part.exists() else 0
            headers={'Range':f'bytes={start}-'} if start else {}
            r=S.get(url,headers=headers,stream=True,timeout=90)
            if r.status_code==416 and part.exists(): part.replace(target); return True
            r.raise_for_status()
            mode='ab' if start else 'wb'
            with open(part,mode) as f:
                for c in r.iter_content(1024*1024):
                    if c: f.write(c)
            part.replace(target); return True
        except Exception as e:
            print('retry',attempt+1,url,str(e))
            time.sleep(min(2**attempt,30))
    return False

def jobs_for():
    jobs=[]
    for interval in ('1m','5m'):
        for m in iter_months(START,END):
            s=f'{m.year:04d}-{m.month:02d}'
            url=f'{BASE}/monthly/klines/{SYMBOL}/{interval}/{SYMBOL}-{interval}-{s}.zip'
            jobs.append(('kline',interval,s,url,CACHE/'klines'/interval/f'{s}.zip'))
    for m in iter_months(START,END):
        s=f'{m.year:04d}-{m.month:02d}'
        url=f'{BASE}/monthly/aggTrades/{SYMBOL}/{SYMBOL}-aggTrades-{s}.zip'
        jobs.append(('aggTrades',None,s,url,CACHE/'aggTrades'/f'{s}.zip'))
    # FIX: futures metrics are DAILY archives.
    for d in iter_days(START,END):
        s=f'{d.year:04d}-{d.month:02d}-{d.day:02d}'
        url=f'{BASE}/daily/metrics/{SYMBOL}/{SYMBOL}-metrics-{s}.zip'
        jobs.append(('metrics',None,s,url,CACHE/'metrics'/f'{s}.zip'))
    return jobs

def do_job(j):
    typ,interval,period,url,path=j
    if path.exists() and path.stat().st_size>100:
        return {'type':typ,'interval':interval,'period':period,'url':url,'path':str(path),'status':'cached','sha256':sha256(path)}
    ok=download(url,path)
    return {'type':typ,'interval':interval,'period':period,'url':url,'path':str(path),'status':'downloaded' if ok else 'not_found_or_failed','sha256':sha256(path) if ok else None}

jobs=jobs_for(); print('Planned objects:',len(jobs))
results=[]
with ThreadPoolExecutor(max_workers=12) as ex:
    futs=[ex.submit(do_job,j) for j in jobs]
    for i,f in enumerate(as_completed(futs),1):
        rec=f.result(); results.append(rec)
        if i%25==0 or rec['status']=='not_found_or_failed': print(i,rec['type'],rec['period'],rec['status'])
json.dump(results,open(REP/'download_manifest.json','w'),indent=2)

# Extract successful archives.
for rec in results:
    p=Path(rec['path'])
    if rec['status'] not in ('downloaded','cached') or not p.exists(): continue
    dest=RAW/Path(p).relative_to(CACHE).parent; dest.mkdir(parents=True,exist_ok=True)
    marker=dest/(p.stem+'.done')
    if marker.exists(): continue
    try:
        with zipfile.ZipFile(p) as z: z.extractall(dest)
        marker.write_text('ok')
    except Exception as e: print('extract error',p,e)

# ---------- KLINES ----------
parts=[]
for p in (RAW/'klines').rglob('*.csv'):
    try:
        d=pd.read_csv(p,header=None,low_memory=False)
        if d.shape[1]<12: continue
        d=d.iloc[:,:12]; d.columns=['open_ms','open','high','low','close','volume','close_ms','quote_volume','trade_count','taker_buy_base','taker_buy_quote','ignore']
        d['ts']=pd.to_datetime(pd.to_numeric(d['open_ms'],errors='coerce'),unit='ms',utc=True)
        for c in ['open','high','low','close','volume','quote_volume','taker_buy_base','taker_buy_quote','trade_count']: d[c]=pd.to_numeric(d[c],errors='coerce')
        parts.append(d[['ts','open','high','low','close','volume','quote_volume','trade_count','taker_buy_base','taker_buy_quote']])
    except Exception: pass
if not parts: raise RuntimeError('No usable Binance kline archives were obtained.')
k=pd.concat(parts,ignore_index=True).drop_duplicates('ts').sort_values('ts')
k=k[(k.ts>=START)&(k.ts<=END)].copy()
k['delta_taker_base']=2*k['taker_buy_base']-k['volume']
k['ret_5m']=k['close'].pct_change()
k['flow_z_4h']=(k['delta_taker_base']-k['delta_taker_base'].rolling(48).mean())/k['delta_taker_base'].rolling(48).std()
k['rv_4h']=k['ret_5m'].rolling(48).std()
k['fwd_ret_5m']=k['close'].shift(-1)/k['close']-1
k['fwd_ret_15m']=k['close'].shift(-3)/k['close']-1
k['fwd_ret_60m']=k['close'].shift(-12)/k['close']-1

# ---------- METRICS / OI / CROWDING ----------
m_parts=[]
for p in (RAW/'metrics').rglob('*.csv'):
    try:
        d=pd.read_csv(p,low_memory=False); d.columns=[str(c).strip() for c in d.columns]
        tc=next((c for c in d.columns if c.lower()=='create_time'),None)
        if tc is None: continue
        d['ts']=pd.to_datetime(pd.to_numeric(d[tc],errors='coerce'),unit='ms',utc=True)
        ren={}
        for c in d.columns:
            q=str(c).lower()
            if q=='sum_open_interest': ren[c]='oi'
            elif 'count_toptrader_long_short_ratio' in q: ren[c]='top_ls_count'
            elif 'sum_toptrader_long_short_ratio' in q: ren[c]='top_ls_position'
            elif 'count_long_short_ratio' in q: ren[c]='global_ls_count'
            elif 'sum_taker_long_short_vol_ratio' in q: ren[c]='taker_ls'
        d=d.rename(columns=ren)
        keep=['ts']+[c for c in ['oi','top_ls_count','top_ls_position','global_ls_count','taker_ls'] if c in d.columns]
        if len(keep)>1: m_parts.append(d[keep])
    except Exception: pass
if m_parts:
    m=pd.concat(m_parts,ignore_index=True).drop_duplicates('ts').sort_values('ts')
    m=m[(m.ts>=START)&(m.ts<=END)]
    k=k.merge(m,on='ts',how='left')
    if 'oi' in k:
        k['oi_change_5m']=k['oi'].diff()
        k['oi_change_1h']=k['oi'].diff(12)
        k['oi_change_4h']=k['oi'].diff(48)
        k['oi_pct_change_1h']=k['oi'].pct_change(12)

# ---------- AGG TRADES ----------
a_parts=[]
for p in (RAW/'aggTrades').rglob('*.csv'):
    try:
        d=pd.read_csv(p,header=None,low_memory=False)
        if d.shape[1]<7: continue
        d=d.iloc[:,:7]; d.columns=['id','price','qty','first_id','last_id','ms','buyer_maker']
        d['ts']=pd.to_datetime(pd.to_numeric(d['ms'],errors='coerce'),unit='ms',utc=True)
        d['qty']=pd.to_numeric(d['qty'],errors='coerce'); d['signed_qty']=d['qty'].where(~d['buyer_maker'],-d['qty']); d['bucket']=d['ts'].dt.floor('5min')
        a_parts.append(d[['id','qty','signed_qty','bucket']])
    except Exception: pass
if a_parts:
    a=pd.concat(a_parts,ignore_index=True).drop_duplicates('id')
    af=a.groupby('bucket').agg(agg_delta=('signed_qty','sum'),agg_trades=('id','size'),avg_trade_size=('qty','mean')).reset_index().rename(columns={'bucket':'ts'})
    k=k.merge(af,on='ts',how='left')

# ---------- QUALITY ----------
k=k.sort_values('ts').reset_index(drop=True)
u=k['ts'].drop_duplicates(); diff=u.diff().dt.total_seconds().div(60)
g=pd.DataFrame({'from_ts':u.iloc[:-1].to_numpy(),'to_ts':u.iloc[1:].to_numpy(),'gap_minutes':diff.iloc[1:].to_numpy()}); g=g[g.gap_minutes>5]
g.to_csv(REP/'gaps.csv',index=False)
quality={'rows':int(len(k)),'start':str(k.ts.min()),'end':str(k.ts.max()),'duplicate_timestamps':int(k.ts.duplicated().sum()),'gap_count_gt_5m':int(len(g)),'null_close':int(k.close.isna().sum()),'oi_non_null_rows':int(k.oi.notna().sum()) if 'oi' in k else 0,'agg_trade_buckets':int(k.agg_delta.notna().sum()) if 'agg_delta' in k else 0}
json.dump(quality,open(REP/'quality.json','w'),indent=2)
k.to_parquet(CAN/'btcusdt_futures_5m.parquet',index=False,compression='zstd')
print('QUALITY\n',json.dumps(quality,indent=2))
print('CANONICAL',CAN/'btcusdt_futures_5m.parquet')


In [ ]:
from pathlib import Path
import zipfile, os
root=Path('/content/btcusdt_research_v2'); pkg=Path('/content/btcusdt_microstructure_research_package.zip')
with zipfile.ZipFile(pkg,'w',zipfile.ZIP_DEFLATED) as z:
    for p in root.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(root))
print('PACKAGE',pkg,'SIZE_MB',round(pkg.stat().st_size/1024/1024,2))
from google.colab import files
files.download(str(pkg))
